In [2]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1cf20809-c383-4141-9ece-a4c70f97c44a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

In [3]:
# ⬇️ Параметры подключения к PostgreSQL public.shops 

jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name = "public.shops"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")


shops_df = (
        spark
        .read
        .format("jdbc")
		.option("url", jdbc_url)
		.option("user", db_user)
		.option("password", db_password)
		.option("dbtable", table_name)
		.option("fetchsize", 1000)
		.option("driver", "org.postgresql.Driver")
		.load()
        )


shops_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [4]:
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
table_name = "public.shop_timezone"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")


shop_timezone_df = (
        spark
        .read
        .format("jdbc")
		.option("url", jdbc_url)
		.option("user", db_user)
		.option("password", db_password)
		.option("dbtable", table_name)
		.option("fetchsize", 1000)
		.option("driver", "org.postgresql.Driver")
		.load()
        )


shop_timezone_df.show()

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [5]:
# Регистрируем DataFrame-ы как временные вьюхи 
shops_df.createOrReplaceTempView("shops") 
shop_timezone_df.createOrReplaceTempView("shop_timezone")
 
# Выполняем SQL запрос для трансформации 
spark.sql(""" with 
sz as (SELECT cast(plant as INT) as id, time_zone FROM shop_timezone where cast(plant as INT) IS NOT null and plant regexp '^[0-9]+$'),

s as (select cast(st_id AS INT) as st_id, shop_name from shops),

cte as (select *, row_number() over(PARTITION BY st_id ORDER BY time_zone desc) as rnk from s join sz ON s.st_id = sz.id)
 
select st_id, shop_name, CAST( CASE WHEN time_zone = '' OR time_zone IS NULL THEN '3' ELSE substr(time_zone, 4) END AS INT)
		AS tz_code from cte where rnk = 1 """).show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [20]:
from pyspark.sql import Window
from pyspark.sql import functions as F

sz = (
    shop_timezone_df
    .where(F.col('plant').rlike('^[0-9]+$'))
    .withColumnRenamed('plant','id')
    .withColumn('id',F.col('id').cast('int').alias('id'))
)
s=(
   shops_df
   .withColumn('st_id',F.col('st_id').cast('int')) 
)
cte=(
    s
    .join(sz, s.st_id==sz.id)
    .withColumn('rnk', F.row_number().over(Window.partitionBy('st_id').orderBy(F.col('time_zone').desc())))
)
res=(
    cte
    .where(F.col('rnk')==1)
    .withColumn('time_zone',F.when((F.col('time_zone')=='' )| (F.col('time_zone').isNull()),3).otherwise(F.col('time_zone').substr(4,10)).cast('int'))
    .select('st_id','shop_name','time_zone')
)
# sz.printSchema()
# sz.show()
# s.printSchema()
# s.show()
# cte.printSchema()
# cte.show()
res.printSchema()
res.show()

root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)
 |-- time_zone: integer (nullable = true)

+-----+-----------+---------+
|st_id|  shop_name|time_zone|
+-----+-----------+---------+
|  842|      Lenta|        7|
|  843|     Magnit|        4|
|  844|       Spar|        3|
|  845|Pyaterochka|        5|
|  847|      Diksi|        3|
|  848|      Lenta|        8|
+-----+-----------+---------+



In [ ]:
spark.stop()